### NAME: MUHAMMAD DANYAL KHALIL
### REG NO: 22JZELE0473
### UNIVERSITY: UET PESHAWAR (NOWSHERA CAMPUS)
### DEPARTMENT: ELECTRICAL (POWER)
# Lab 12: LSTM for Time-Series Forecasting

**Course:** Machine Learning Lab  
**Topic:** Long Short-Term Memory network for AEP forecasting

## Lab Objective
The objective of this lab is to build and train an LSTM model for forecasting time-series values. LSTM layers are used because they can learn sequential dependencies across time steps.


## 1. Set Working Directory
The working directory is set so the notebook can access local datasets, helper modules, checkpoints, and saved training outputs.


In [35]:
import os
os.chdir(r'H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12\Code')

## 2. Import Libraries and Utilities
This cell imports forecasting metrics, sequence preparation functions, TensorFlow/Keras layers, callbacks, plotting tools, and scaling utilities.


In [36]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

## 3. Define Initial Parameters
The time-window length, number of features, starting epoch, and model variable are initialized before creating the LSTM network.


In [37]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

## 4. Define the LSTM Architecture
The LSTM model processes sequential input data using stacked LSTM layers, then outputs one predicted value through a dense layer.


In [38]:
def create_lstm():
    input_data = Input(shape=(time_steps, num_features))
    lstm_layer1 = LSTM(8, return_sequences=True)(input_data)
    lstm_layer2 = LSTM(20)(lstm_layer1)
    x = Flatten()(lstm_layer2)
    output_data = Dense(1)(x)
    model = Model(input_data, output_data)
    return model

## 5. Display Model Summary and Diagram
The model summary and diagram verify the LSTM architecture, input shape, and trainable parameters.


In [39]:
model1 = create_lstm()
model1.summary()

Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_5 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 lstm_8 (LSTM)               (None, 24, 8)             960       
                                                                 
 lstm_9 (LSTM)               (None, 20)                2320      
                                                                 
 flatten_4 (Flatten)         (None, 20)                0         
                                                                 
 dense_4 (Dense)             (None, 1)                 21        
                                                                 
Total params: 3,301
Trainable params: 3,301
Non-trainable params: 0
_________________________________________________________________


In [40]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


## 6. Set Checkpoint and History Paths
These paths define where the best LSTM model and training history will be saved.


In [41]:
checkpoints = r'H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

## 7. Configure Callbacks
The checkpoint callback saves the best validation-loss model, and the training monitor records training progress.


In [42]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

## 8. Compile or Load the LSTM
A new LSTM is compiled when no model checkpoint is supplied. Otherwise, the saved model is loaded and training can continue.


In [43]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =create_lstm()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## 9. Load Dataset Files
Training, validation, and testing CSV files are loaded from the processed dataset folder. The scaler is loaded for converting predictions back to original units.


In [54]:
import os
path_dataset =(r'H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12\Code')
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Lenovo\anaconda3\envs\ml_env\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [45]:
time_steps=24
num_features=21

## 10. Prepare Sequential Data
The time-step configuration is confirmed, then the dataset is converted into supervised sequence windows for the LSTM.


In [46]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.8440830707550049 sec


## 11. Train the LSTM
The LSTM is trained on sequential windows and validated after each epoch. Checkpoints are saved based on validation loss.


In [47]:
epochs = 2
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/2
2652/2653 [============================>.] - ETA: 0s - loss: 0.0417 - mae: 0.0417 - mape: 713.1125
Epoch 1: val_loss improved from inf to 0.01935, saving model to H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12\E1-cp-0001-loss0.02.h5
2653/2653 [==============================] - 99s 34ms/step - loss: 0.0417 - mae: 0.0417 - mape: 712.9625 - val_loss: 0.0193 - val_mae: 0.0193 - val_mape: 8.3143
Epoch 2/2
2652/2653 [============================>.] - ETA: 0s - loss: 0.0165 - mae: 0.0165 - mape: 313.8183
Epoch 2: val_loss improved from 0.01935 to 0.01263, saving model to H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12\E1-cp-0002-loss0.01.h5
2653/2653 [==============================] - 90s 34ms/step - loss: 0.0165 - mae: 0.0165 - mape: 313.7522 - val_loss: 0.0126 - val_mae: 0.0126 - val_mape: 6.6388


## 12. Evaluate Test Performance
The best saved model is loaded, test predictions are inverse-transformed, and MAE, RMSE, MAPE, and R2 are calculated.


In [49]:

model = load_model(r'H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12\E1-cp-0001-loss0.01.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 [==============================] - 5s 10ms/step
Mean Absolute Error (MAE): 212.39
Median Absolute Error (MedAE): 180.56
Mean Squared Error (MSE): 71035.13
Root Mean Squared Error (RMSE): 266.52
Mean Absolute Percentage Error (MAPE): 1.48 %
Median Absolute Percentage Error (MDAPE): 1.24 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


## 13. Fine-Tuning Setup
This cell prepares a second training stage by selecting a previous model checkpoint and setting the starting epoch.


In [50]:
checkpoints = r'H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12\E1-cp-0001-loss0.01.h5'
start_epoch= 5

## 14. Rebuild Callbacks and Load Model
Callbacks and the model are prepared again so fine-tuning can continue from the saved checkpoint.


In [51]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12\E1-cp-0001-loss0.01.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


## 15. Continue Training
The LSTM is trained for additional epochs to refine its forecasting performance.


In [52]:
epochs = 2
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/2
2653/2653 [==============================] - ETA: 0s - loss: 0.0127 - mae: 0.0127 - mape: 407.6506
Epoch 1: val_loss improved from inf to 0.01169, saving model to H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12\E2-cp-0001-loss0.01.h5
2653/2653 [==============================] - 102s 37ms/step - loss: 0.0127 - mae: 0.0127 - mape: 407.6506 - val_loss: 0.0117 - val_mae: 0.0117 - val_mape: 5.6167
Epoch 2/2
2652/2653 [============================>.] - ETA: 0s - loss: 0.0118 - mae: 0.0118 - mape: 333.1252
Epoch 2: val_loss improved from 0.01169 to 0.01152, saving model to H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12\E2-cp-0002-loss0.01.h5
2653/2653 [==============================] - 97s 36ms/step - loss: 0.0118 - mae: 0.0118 - mape: 333.0554 - val_loss: 0.0115 - val_mae: 0.0115 - val_mape: 5.9681


## 16. Evaluate Fine-Tuned LSTM
The fine-tuned model is evaluated on the test set using the same metrics for a fair comparison.


In [53]:

model = load_model(r'H:\DRIVE D DATA\UET\8th Semester\Machine Learning Lab\lab no 12\E2-cp-0001-loss0.01.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 [==============================] - 5s 10ms/step
Mean Absolute Error (MAE): 179.79
Median Absolute Error (MedAE): 144.73
Mean Squared Error (MSE): 53434.75
Root Mean Squared Error (RMSE): 231.16
Mean Absolute Percentage Error (MAPE): 1.24 %
Median Absolute Percentage Error (MDAPE): 1.0 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


## Conclusion
This lab demonstrates the use of LSTM networks for time-series forecasting. LSTMs are suitable for sequential data because they can learn dependencies across previous time steps, and fine-tuning helps improve a saved model.
